# RQ1 Statistics — Python replication

Reproduces, for **Python**, the numbers reported in the Java RQ1 paragraph of the Stack Overflow answer-edit study:

- number of accepted answers (baseline denominator)
- number of code snippets + avg/std per answer (latest version)
- number of answers with ≥ 1 revision + percentage (the filtered set)
- average / median / max revisions per answer
- total code snippets across all revisions
- the answer with the most revisions (+ its date span)
- a revisions-distribution histogram (`post_revisions_histogram.pdf`)

The filtered set is read from the answer-id file produced by `extract_answer_list.py`.
Baseline figures are computed over *all* accepted answers for the tag.

Run the cells in order. Configure the connection and paths in the **Configuration** cell.

In [ ]:
!pip install mysql-connector-python matplotlib

## Configuration

In [ ]:
import statistics
import time
from pathlib import Path

import mysql.connector
import matplotlib.pyplot as plt

# --- connection ---
HOST = "127.0.0.1"
USER = "root"
PASSWORD = ""
DATABASE = "sotorrent"

# --- study parameters (assumes the notebook runs from the analysis/ directory) ---
TAG = "python"
ANSWER_LIST = Path("../files/acceptedWithVersionAnswer_python.txt")  # output of extract_answer_list.py
HIST_OUTPUT = Path("post_revisions_histogram.pdf")
SKIP_BASELINE = False  # set True to skip the slow all-accepted-answers baseline queries

INSERT_BATCH = 20_000

## Connect and resolve the tag id

In [ ]:
db = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
cursor = db.cursor()

cursor.execute("SELECT Id, Count FROM Tags WHERE TagName = %s", (TAG,))
row = cursor.fetchone()
if row is None:
    raise ValueError(f"Tag '{TAG}' not found in the Tags table.")
tag_id, tag_count = row
print(f"Tag '{TAG}' -> TagId {tag_id} ({tag_count:,} questions)")

## Baseline: all accepted answers for the tag

Builds a temporary table of every accepted answer for the tag, then counts latest-version code snippets per answer (including answers with zero code blocks). Skipped if `SKIP_BASELINE = True`. These queries touch the full accepted set, so they can take a few minutes.

In [ ]:
baseline = None
if not SKIP_BASELINE:
    t0 = time.perf_counter()
    cursor.execute("DROP TEMPORARY TABLE IF EXISTS _rq1_all_accepted")
    cursor.execute("CREATE TEMPORARY TABLE _rq1_all_accepted (Id INT PRIMARY KEY)")
    cursor.execute(
        """
        INSERT IGNORE INTO _rq1_all_accepted (Id)
        SELECT q.AcceptedAnswerId
        FROM Posts q
        JOIN PostTags pt ON pt.PostId = q.Id AND pt.TagId = %s
        WHERE q.PostTypeId = 1 AND q.AcceptedAnswerId IS NOT NULL
        """,
        (tag_id,),
    )
    cursor.execute("SELECT COUNT(*) FROM _rq1_all_accepted")
    total_accepted = cursor.fetchone()[0]
    print(f"accepted answers: {total_accepted:,}  ({time.perf_counter() - t0:.0f}s)")

    print("counting latest-version code snippets per answer (may take minutes)...")
    t0 = time.perf_counter()
    cursor.execute(
        """
        SELECT COUNT(*)              AS answers,
               COALESCE(SUM(cnt), 0) AS snippets,
               AVG(cnt)              AS avg_snippets,
               STDDEV_SAMP(cnt)      AS std_snippets
        FROM (
            SELECT aa.Id, COUNT(pbv.Id) AS cnt
            FROM _rq1_all_accepted aa
            LEFT JOIN PostBlockVersion pbv
                   ON pbv.PostId = aa.Id
                  AND pbv.PostBlockTypeId = 2
                  AND pbv.MostRecentVersion = 1
            GROUP BY aa.Id
        ) x
        """
    )
    _, snippets, avg_snip, std_snip = cursor.fetchone()
    baseline = {
        "total_accepted": total_accepted,
        "total_snippets": int(snippets),
        "avg_snippets": float(avg_snip),
        "std_snippets": float(std_snip),
    }
    print(f"code snippets: {baseline['total_snippets']:,}  "
          f"avg {baseline['avg_snippets']:.2f}/answer, std {baseline['std_snippets']:.2f}  "
          f"({time.perf_counter() - t0:.0f}s)")
else:
    print("baseline skipped")

## Filtered set: load the answer-id list

Loads the ids produced by `extract_answer_list.py` into a temporary table.

In [ ]:
if not ANSWER_LIST.exists():
    raise FileNotFoundError(f"Answer list not found: {ANSWER_LIST}\nRun extract_answer_list.py first.")

ids = [int(line) for line in ANSWER_LIST.read_text().split() if line.strip()]
filtered_total = len(ids)
print(f"loading {filtered_total:,} answer ids from {ANSWER_LIST.name}...")

cursor.execute("DROP TEMPORARY TABLE IF EXISTS _rq1_filtered")
cursor.execute("CREATE TEMPORARY TABLE _rq1_filtered (Id INT PRIMARY KEY)")
for i in range(0, len(ids), INSERT_BATCH):
    batch = ids[i:i + INSERT_BATCH]
    cursor.executemany("INSERT IGNORE INTO _rq1_filtered (Id) VALUES (%s)", [(x,) for x in batch])
db.commit()
print("loaded")

## Filtered set: revision counts, snippets, and the most-revised answer

In [ ]:
print("counting versions per answer...")
cursor.execute(
    """
    SELECT pv.PostId, COUNT(*) AS versions
    FROM PostVersion pv
    JOIN _rq1_filtered t ON t.Id = pv.PostId
    GROUP BY pv.PostId
    """
)
counts = {post_id: versions for post_id, versions in cursor.fetchall()}
revisions = list(counts.values())

print("counting code snippets across all revisions...")
cursor.execute(
    """
    SELECT COUNT(*)
    FROM PostBlockVersion pbv
    JOIN _rq1_filtered t ON t.Id = pbv.PostId
    WHERE pbv.PostBlockTypeId = 2
    """
)
snippets_all_revisions = cursor.fetchone()[0]

most_revised_id = max(counts, key=counts.get)
cursor.execute(
    "SELECT MIN(CreationDate), MAX(CreationDate) FROM PostVersion WHERE PostId = %s",
    (most_revised_id,),
)
most_revised_first, most_revised_last = cursor.fetchone()
print("done")

## Revisions-distribution histogram

In [ ]:
max_rev = max(revisions)
bins = range(min(revisions), max_rev + 2)
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(revisions, bins=bins, color="#4d4d4d", edgecolor="black", linewidth=0.5)
ax.set_xlabel("Number of revisions", fontsize=14)
ax.set_ylabel("Number of answers", fontsize=14)
ax.set_yscale("log")
ax.tick_params(axis="both", which="major", labelsize=12)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(HIST_OUTPUT, dpi=300, bbox_inches="tight", format="pdf")
print(f"Histogram saved to: {HIST_OUTPUT}")
plt.show()

## RQ1 summary

In [ ]:
print("=" * 70)
print(f"RQ1 STATISTICS - tag '{TAG}'")
print("=" * 70)
if baseline:
    pct = 100.0 * filtered_total / baseline["total_accepted"]
    print(f"Accepted answers:                 {baseline['total_accepted']:,}")
    print(f"Code snippets (latest version):   {baseline['total_snippets']:,}")
    print(f"  avg per answer:                 {baseline['avg_snippets']:.2f}")
    print(f"  std:                            {baseline['std_snippets']:.2f}")
    print(f"Answers with >= 1 revision:       {filtered_total:,} ({pct:.2f}%)")
else:
    print(f"Answers with >= 1 revision:       {filtered_total:,} (baseline skipped -> no percentage)")
print(f"Avg revisions per answer:         {statistics.mean(revisions):.2f}")
print(f"Median revisions per answer:      {statistics.median(revisions):.1f}")
print(f"Std revisions per answer:         {statistics.pstdev(revisions):.2f}")
print(f"Max revisions:                    {counts[most_revised_id]} (post ID {most_revised_id})")
print(f"  first revision:                 {most_revised_first}")
print(f"  last revision:                  {most_revised_last}")
print(f"Code snippets across all revs:    {snippets_all_revisions:,}")
print("=" * 70)

## Close the connection

In [ ]:
cursor.close()
db.close()
print("connection closed")